In [1]:
! pip install openai

In [2]:
import os
from openai import OpenAI

In [5]:
from openai import OpenAI
from google.colab import userdata

# Get the Groq API key from Colab's secrets manager
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Ensure the API key is not None
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

response = client.responses.create(
    input="Explain the importance of fast language models",
    model="openai/gpt-oss-20b",
)
print(response.output_text)

**Fast language models**—those that can generate high‑quality text in a fraction of a second—are becoming a cornerstone of modern AI because they unlock a host of practical, economic, and environmental benefits. Below is a quick‑look on why speed matters, the trade‑offs involved, and how it shapes the future of AI‑powered services.

| # | Why Speed Is Crucial | What It Enables |
|---|----------------------|-----------------|
| **1. Real‑time user experience** | Chatbots, virtual assistants, and live translation tools need to respond in *milliseconds* to feel conversational. A 200‑ms latency is often perceived as instant, whereas 1–2 s can break the flow. | • Seamless customer support<br>• Interactive language learning apps<br>• Voice‑activated devices |
| **2. Scalable deployment** | Large models (e.g., 175 B parameters) can cost millions of dollars in GPU hours for a single inference request. Faster models cut compute per request, letting you serve **thousands of requests per GPU**. |

In [6]:
! pip install langchain faiss-cpu langchain_community tiktoken sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is 

In [7]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/company_dataset.txt')
documents=loader.load()

/tmp/ipykernel_3202/3572841172.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [8]:
# step 2 create the embedding + vector DB
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the HuggingFace embeddings
# Using a common open-source model; ensure 'sentence-transformers' is installed
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Create the vector database from your pre-loaded documents
vector_db = FAISS.from_documents(documents, embeddings)

/tmp/ipykernel_3202/4207546490.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# #step 3 Retrieval + Generation
def rag_query(query):
    # Retrieve the top 3 most relevant document chunks based on the query
    docs = vector_db.similarity_search(query, k=3)

    # Combine the content of the retrieved documents into a single context string
    context = " ".join([doc.page_content for doc in docs])

    # Generate the response using OpenAI's chat completions API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model to the user-specified Groq model
        messages=[
            {"role": "system", "content": "use the provided context to answer the question"},
            {"role": "user", "content": f"context: {context}\n\nquery: {query}"}
        ]
    )

    return response.choices[0].message.content

In [10]:
print(rag_query("What is the refund policy of TechNova Solutions Pvt Ltd?"))

The refund policy of TechNova Solutions Pvt Ltd is that customers can request a refund within 14 days of purchase, and refunds are processed within 7-10 business days.


In [11]:
import warnings
warnings.filterwarnings('ignore')
!pip install bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [12]:
# #step 1 : Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# #define the bitsandbytes config for 8 bit quantization
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [13]:
# #step 2 : Apply LoRA(PEFT)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# #preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)

In [14]:
from datasets import Dataset

data = [
    {
        "text": "Q: What is TechNova Solutions Pvt Ltd?\nA: TechNova Solutions is a technology company providing software development, cloud services, AI solutions, and IT consulting."
    },

    {
        "text": "Q: What is the refund policy of TechNova Solutions Pvt Ltd?\nA: Customers can request a refund within 14 days of purchase. Refunds are processed within 7-10 business days."
    },

    {
        "text": "Q: What is the return policy of TechNova Solutions Pvt Ltd?\nA: Physical products can be returned within 15 days if unopened."
    },

    {
        "text": "Q: What are the shipping options available at TechNova Solutions Pvt Ltd?\nA: Standard delivery takes 5-7 business days and express delivery takes 2-3 business days."
    },

    {
        "text": "Q: How can I contact TechNova Solutions customer support?\nA: Email: support@technova.com, Phone: +91-9123456789."
    },

    {
        "text": "Q: What are the business hours of TechNova Solutions?\nA: Monday to Friday: 9 AM to 6 PM."
    },

    {
        "text": "Q: Does TechNova Solutions protect customer data?\nA: Yes, customer data is encrypted and protected."
    },

    {
        "text": "Q: Does TechNova Solutions have a loyalty program?\nA: Yes, customers earn reward points for purchases."
    },

    {
        "text": "Q: What is GreenMart Online Pvt Ltd?\nA: GreenMart Online specializes in organic food and eco-friendly products."
    },

    {
        "text": "Q: What is the refund policy of GreenMart Online Pvt Ltd?\nA: Customers may request refunds within 10 days of delivery."
    },

    {
        "text": "Q: What is the return policy of GreenMart Online Pvt Ltd?\nA: Unused products can be returned within 12 days."
    },

    {
        "text": "Q: What are the shipping options available at GreenMart Online?\nA: Standard delivery takes 4-6 business days and express delivery takes 1-2 business days."
    },

    {
        "text": "Q: How can I contact GreenMart Online customer support?\nA: Email: help@greenmart.com, Phone: +91-9988776655."
    },

    {
        "text": "Q: What are the business hours of GreenMart Online?\nA: Monday to Sunday: 8 AM to 10 PM."
    },

    {
        "text": "Q: Does GreenMart Online protect customer data?\nA: Yes, customer information is protected using encrypted databases."
    },

    {
        "text": "Q: Does GreenMart Online have a loyalty program?\nA: Yes, customers earn eco-points for every purchase."
    },

    {
        "text": "Q: What is FreshBasket Online Pvt Ltd?\nA: FreshBasket delivers groceries, fruits, vegetables, and household essentials."
    },

    {
        "text": "Q: What is the refund policy of FreshBasket Online Pvt Ltd?\nA: Refund requests are accepted within 5 days of delivery."
    },

    {
        "text": "Q: What is the return policy of FreshBasket Online Pvt Ltd?\nA: Damaged products may be returned within 7 days."
    },

    {
        "text": "Q: What are the shipping options available at FreshBasket Online?\nA: Standard delivery takes 2-4 business days and express delivery takes 1 day."
    },

    {
        "text": "Q: How can I contact FreshBasket customer support?\nA: Email: support@freshbasket.com, Phone: +91-9876501234."
    },

    {
        "text": "Q: What are the business hours of FreshBasket Online?\nA: Monday-Sunday: 7 AM - 10 PM."
    },

    {
        "text": "Q: Does FreshBasket protect customer data?\nA: Yes, customer data is encrypted and protected."
    },

    {
        "text": "Q: Does FreshBasket have a loyalty program?\nA: Yes, customers earn 2 reward points for every ₹100 spent."
    }
]

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 24
})


In [15]:
# step 4: tokenization
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True,padding="max_length",
    max_length=128)


tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

In [16]:
from transformers import Trainer , TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)

# Defensive check and re-application of PEFT if model is not recognized as a PeftModel
# This ensures that the model is correctly prepared for PEFT training
if not isinstance(model, PeftModel):
    print("Warning: Model not recognized as a PEFT model. Attempting to re-apply PEFT configuration.")
    # Redefine lora_config to ensure it's available in this scope
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.5,
        bias="none",
        task_type="CAUSAL_LM"
    )
    # The 'model' variable is assumed to be the base quantized model at this point if it's not a PeftModel.
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    print("PEFT configuration re-applied.")

training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Step,Training Loss
10,7.693459
20,3.979042
30,2.713591


TrainOutput(global_step=30, training_loss=4.7953638712565105, metrics={'train_runtime': 63.2128, 'train_samples_per_second': 1.898, 'train_steps_per_second': 0.475, 'total_flos': 95444470333440.0, 'train_loss': 4.7953638712565105, 'epoch': 5.0})

In [20]:
def generate_response(question):
    prompt = f"Q: {question}\nA:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [21]:
print(generate_response("What is TechNova Solutions Pvt Ltd?"))
print(generate_response("What is GreenMart Online Pvt Ltd?"))
print(generate_response("What is FreshBasket Online Pvt Ltd?"))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is TechNova Solutions Pvt Ltd?
A: Teen, and the same time.












































[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is GreenMart Online Pvt Ltd?
A: Greenwood, and the same time.










































Q: What is FreshBasket Online Pvt Ltd?
A: Fair and the same time.











































